# Runs comparer

Cross-experiment comparison of sweep runs: metrics heatmap, covering-rank stats,
gold-rank table, late golds. Runs are loaded from every experiment's `runs/` folder;
select which to compare by label.

In [1]:
from pathlib import Path
import pandas as pd

from minerva_eval import load_all_results, get_run_summary, metrics_heatmap, gold_ranks, covering_rank_stats, late_golds

ROOTS = sorted(Path("../experiments").glob("*/runs"))   # every experiment's runs
all_results = load_all_results(ROOTS)

collections = {r.collection for r in all_results}
print(f"{len(all_results)} rows from {len(collections)} collections: {collections}")

420 rows from 2 collections: {'wp1283-legacy', 'wp1283-nollm'}


## Run summary

One row per run folder: collection, timestamp, sweep matrix, query count.

In [2]:
run_summary = get_run_summary(ROOTS)
run_summary

,folder,collection,timestamp,top_k,candidate_pool_size,sweep,queries,rows
0,2026-09-20T07-24-47Z_wp1283-v1,wp1283-nollm,2026-09-20T07:24:47Z,50,100,"alpha=[0.3, 0.5, 0.7], reranker=[False], reran...",84,252
1,2026-09-21T14-31-25Z_wp1283-v1,wp1283-legacy,2026-09-21T14:31:25Z,50,50,"alpha=[1.0], reranker=[False], rerank_depth=No...",84,84
2,2026-09-20T07-30-58Z_wp1283-v1,wp1283-nollm,2026-09-20T07:30:58Z,50,100,"alpha=[0.7], reranker=[True], rerank_depth=[10...",84,84


In [3]:
import ipywidgets as widgets

labels = sorted({r.label for r in all_results})
picker = widgets.SelectMultiple(options=labels, value=tuple(labels),
                                description="runs", rows=(len(labels) + 2), layout=widgets.Layout(width="700px"))
display(picker)

SelectMultiple(description='runs', index=(0, 1, 2), layout=Layout(width='700px'), options=('baseline', 'minerv…

## Metrics view

Mean over queries, grouped by `(label, collection, hybrid_alpha, reranker_depth)`. Reproduces the
headline table from the sweep-findings spec; styled as a heatmap.

In [4]:
chosen = [r for r in all_results if r.label in picker.value]
metrics_heatmap(chosen)

## Covering rank

Covering rank = smallest $k$ whose top-$k$ contains **all** the query's golds
(max over gold sources), so it is the depth a client has to read to get the
whole answer. One row per cell: `p90` = 4 reads as "90% of queries have every
gold within top 4". `p50` is 1 almost everywhere and says nothing; the cells
differ in the tail. `beyond` counts the queries whose gold never appeared
inside `top_k` (the sentinel `top_k + 1`), kept out of `max` so a sentinel
cannot pass for a rank.

The gold rank table below breaks these numbers down query by query.

In [5]:
covering_rank_stats(chosen)

n  p50  p90  p95  \
label    collection    alpha rerank_depth cascade_depth                      
baseline wp1283-nollm  0.3   0            0              84    1   14   35   
                       0.5   0            0              84    1    5    7   
                       0.7   0            0              84    1    4    5   
minerva1 wp1283-legacy 1.0   0            0              84    1    5    8   
reranker wp1283-nollm  0.7   100          20             84    1    2    3   

                                                         max  beyond  
label    collection    alpha rerank_depth cascade_depth               
baseline wp1283-nollm  0.3   0            0               48       0  
                       0.5   0            0               15       0  
                       0.7   0            0               12       0  
minerva1 wp1283-legacy 1.0   0            0               24       0  
reranker wp1283-nollm  0.7   100          20               5       0

## Gold rank table

Value is the gold's worst source rank in that cell. Queries with multiple golds
collapse with `max` (the worst-placed gold). Rows are sorted hardest
first (largest worst-case rank on top), so trivially easy queries (rank 1 in
every cell) sink to the bottom. Green = low rank (easy), red = high rank (hard).

In [6]:
ranks = pd.DataFrame(vars(g) for g in gold_ranks(chosen))

table = ranks.pivot_table(
    index="query_id", columns=["collection", "label", "alpha", "rerank_depth", "cascade_depth"],
    values="rank", 
    aggfunc="max"           # multi-gold -> all have been found?
    )

table = table.loc[table.max(axis=1).sort_values(ascending=False).index]

table.style.background_gradient(cmap="RdYlGn_r", axis=None)

## Late golds

One row per gold **source** beyond `threshold`, worst first, so a query that is
deep in the tables above can be read apart: is one source missing, or all of
them? A query with a single gold appears once; a two-gold query appears twice
when both are late. Queries whose golds are all within the threshold are absent.

In [7]:
late_golds(chosen, threshold=10)

,label,collection,alpha,rerank_depth,cascade_depth,query_id,gold,rank
9,baseline,wp1283-nollm,0.3,0,0,The_Beatles.md-2,The_Beatles.md,48
22,baseline,wp1283-nollm,0.3,0,0,List_of_Roman_emperors.md-1,Roman_emperor.md,42
56,baseline,wp1283-nollm,0.3,0,0,Bobcat.md-1,Lynx.md,38
30,baseline,wp1283-nollm,0.3,0,0,Uranium_City.md-1,Uranium_City.md,37
10,baseline,wp1283-nollm,0.3,0,0,The_Beatles.md-3,The_Beatles.md,35
39,baseline,wp1283-nollm,0.3,0,0,Antarctica.md-3,Antarctica.md,30
24,baseline,wp1283-nollm,0.3,0,0,A_Fistful_of_Dollars.md-1,A_Fistful_of_Dollars.md,24
282,minerva1,wp1283-legacy,1.0,0,0,The_Beatles.md-2,The_Beatles.md,24
17,baseline,wp1283-nollm,0.3,0,0,Tharman_Shanmugaratnam.md-1,Tharman_Shanmugaratnam.md,21
23,baseline,wp1283-nollm,0.3,0,0,List_of_Roman_emperors.md-1,List_of_Roman_emperors.md,17
